In [0]:
df = spark.read.option("header", "true").csv("s3://mimic-iv-datasets/mimic-iv-dataset-3.1/icu/chartevents.csv")
print(df.count())

In [0]:
from pyspark.sql.functions import col, lower, when, lit, trim, expr

# Pull straight from bronze — no dependency on clean_omr
omr = spark.table("cdr_bronze.raw_mimic_iv.omr")

hw = (
    omr.withColumn(
        "result_category",
        when(lower(col("result_name")).contains("height"), lit("Height"))
        .when(lower(col("result_name")).contains("weight"), lit("Weight"))
        .otherwise(lit(None))
    )
    .filter(col("result_category").isNotNull())
    .filter(col("result_value").isNotNull() & (trim(col("result_value")) != ""))
    .withColumn("value_num", expr("try_cast(result_value as double)"))
    .filter(col("value_num").isNotNull())
)

# IQR-based outlier bounds, computed separately per category
for category in ["Height", "Weight"]:
    subset = hw.filter(col("result_category") == category)
    q1, q3 = subset.approxQuantile("value_num", [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    print(f"{category}: Q1={q1:.1f}, Q3={q3:.1f}, IQR={iqr:.1f}, "
          f"bounds=({lower_bound:.1f}, {upper_bound:.1f})")

    outliers = subset.filter((col("value_num") < lower_bound) | (col("value_num") > upper_bound))
    print(f"  {category} outlier count: {outliers.count()}")
    display(outliers.orderBy(col("value_num").desc()).limit(20))